# Retail Pricing Intelligence & Dynamic Pricing Simulator

## Phase 4: Data Cleaning & Feature Engineering

In this phase, we transform multiple raw e-commerce datasets into a single analytics-ready dataset.

The objectives are to:

- Load all required datasets
- Validate data quality
- Merge related tables
- Engineer new business features
- Handle missing values and duplicates
- Prepare the data for exploratory analysis, SQL queries, machine learning, and Power BI dashboards

This notebook creates the foundation for all subsequent analyses.

In [1]:
from pathlib import Path
import pandas as pd

## Step 1: Load the Raw Datasets

The Brazilian E-commerce (Olist) dataset is distributed across multiple relational tables.

Each table contains a different aspect of the business, including:

- Customers
- Orders
- Products
- Payments
- Reviews
- Order Items

We load each dataset separately before validating and merging them.

In [2]:
DATA_PATH = Path("../data/raw")

## Step 2: Verify Dataset Dimensions

Before merging datasets, it is important to verify that each table has been loaded successfully.

Checking dataset dimensions helps identify:

- Missing or corrupted files
- Unexpected row counts
- Data loading issues

This simple validation step is standard practice in data engineering and analytics workflows.

In [3]:
orders = pd.read_csv(
    DATA_PATH / "olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

customers = pd.read_csv(DATA_PATH / "olist_customers_dataset.csv")

items = pd.read_csv(DATA_PATH / "olist_order_items_dataset.csv")

products = pd.read_csv(DATA_PATH / "olist_products_dataset.csv")

payments = pd.read_csv(DATA_PATH / "olist_order_payments_dataset.csv")

reviews = pd.read_csv(DATA_PATH / "olist_order_reviews_dataset.csv")

In [4]:
datasets = {
    "orders": orders,
    "customers": customers,
    "items": items,
    "products": products,
    "payments": payments,
    "reviews": reviews
}

for name, df in datasets.items():
    print(f"{name:12} {df.shape}")

orders       (99441, 8)
customers    (99441, 5)
items        (112650, 7)
products     (32951, 9)
payments     (103886, 5)
reviews      (99224, 7)


## Step 3: Build the Master Analytics Dataset

The Olist dataset is stored across multiple relational tables. To perform meaningful business analysis, we combine these tables into a single master dataset.

The merge process follows the relationships between orders, customers, products, payments, reviews, and order items.

This analytical dataset will serve as the primary source for exploratory analysis, feature engineering, SQL queries, predictive modeling, and dashboard development.

In [5]:
master_df = (
    orders
    .merge(customers, on="customer_id", how="left")
    .merge(items, on="order_id", how="left")
    .merge(products, on="product_id", how="left")
    .merge(payments, on="order_id", how="left")
    .merge(reviews, on="order_id", how="left")
)

## Step 4: Inspect the Merged Dataset

After merging the datasets, we verify that the operation completed successfully.

We examine:

- Dataset dimensions
- Sample records
- Column names

This helps confirm that all expected information has been integrated into the analytical dataset.

In [6]:
master_df.shape

(119143, 36)

In [7]:
master_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,payment_sequential,payment_type,payment_installments,payment_value,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,1.0,credit_card,1.0,18.12,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,3.0,voucher,1.0,2.00,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,2.0,voucher,1.0,18.59,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,1.0,boleto,1.0,141.46,8d5266042046a06655c8db133d120ba5,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,1.0,credit_card,3.0,179.12,e73b67b67587f7644d5bd1a52deb1b01,5.0,NaN,NaN,2018-08-18 00:00:00,2018-08-22 19:07:58


In [8]:
master_df.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value',
 'product_category_name',
 'product_name_lenght',
 'product_description_lenght',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm',
 'payment_sequential',
 'payment_type',
 'payment_installments',
 'payment_value',
 'review_id',
 'review_score',
 'review_comment_title',
 'review_comment_message',
 'review_creation_date',
 'review_answer_timestamp']

# Step 5: Assess Data Quality

Before cleaning the data, we evaluate its overall quality.

This includes:

- Missing values
- Duplicate records
- Data types
- Invalid values
- Overall completeness

Understanding these issues helps us decide which cleaning techniques should be applied without accidentally losing useful information.

In [9]:
master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119143 entries, 0 to 119142
Data columns (total 36 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       119143 non-null  str           
 1   customer_id                    119143 non-null  str           
 2   order_status                   119143 non-null  str           
 3   order_purchase_timestamp       119143 non-null  datetime64[us]
 4   order_approved_at              118966 non-null  datetime64[us]
 5   order_delivered_carrier_date   117057 non-null  datetime64[us]
 6   order_delivered_customer_date  115722 non-null  datetime64[us]
 7   order_estimated_delivery_date  119143 non-null  datetime64[us]
 8   customer_unique_id             119143 non-null  str           
 9   customer_zip_code_prefix       119143 non-null  int64         
 10  customer_city                  119143 non-null  str           
 11  customer_st

In [10]:
missing = (
    master_df
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

missing[missing > 0]

review_comment_title             105154
review_comment_message            68898
order_delivered_customer_date      3421
product_category_name              2542
product_name_lenght                2542
product_photos_qty                 2542
product_description_lenght         2542
order_delivered_carrier_date       2086
review_creation_date                997
review_score                        997
review_id                           997
review_answer_timestamp             997
product_height_cm                   853
product_weight_g                    853
product_width_cm                    853
product_length_cm                   853
product_id                          833
order_item_id                       833
freight_value                       833
price                               833
shipping_limit_date                 833
seller_id                           833
order_approved_at                   177
payment_type                          3
payment_sequential                    3


In [11]:
master_df.duplicated().sum()

np.int64(0)

In [12]:
master_df.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
customer_unique_id                          str
customer_zip_code_prefix                  int64
customer_city                               str
customer_state                              str
order_item_id                           float64
product_id                                  str
seller_id                                   str
shipping_limit_date                         str
price                                   float64
freight_value                           float64
product_category_name                       str
product_name_lenght                     float64
product_description_lenght              

In [13]:
master_df.describe(include="all")

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,payment_sequential,payment_type,payment_installments,payment_value,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
count,119143,119143,119143,119143,118966,117057,115722,119143,119143,119143.000000,...,119140.000000,119140,119140.000000,119140.000000,118146,118146.000000,13989,50245,118146,118146
unique,99441,99441,8,NaN,NaN,NaN,NaN,NaN,96096,NaN,...,NaN,5,NaN,NaN,98410,NaN,4527,36159,636,98248
top,895ab968e7bb0d5659d16cd74cd1650c,270c23a11d024a44c896d1894b261a83,delivered,NaN,NaN,NaN,NaN,NaN,9a736b248f67d166d2fbb006bcb877c3,NaN,...,NaN,credit_card,NaN,NaN,eef5dbca8d37dfce6db7d7b16dd0525e,NaN,Recomendo,Muito bom,2017-12-19 00:00:00,2017-08-17 22:17:55
freq,63,63,115723,NaN,NaN,NaN,NaN,NaN,75,NaN,...,NaN,87776,NaN,NaN,63,NaN,494,259,547,63
mean,NaN,NaN,NaN,2017-12-29 18:36:13.115760,2017-12-30 04:49:18.425726,2018-01-03 08:24:34.395525,2018-01-12 20:55:38.199616,2018-01-22 15:21:10.241642,NaN,35033.451298,...,1.094737,NaN,2.941246,172.735135,NaN,4.015582,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,2016-09-04 21:15:19,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-09-30 00:00:00,NaN,1003.000000,...,1.000000,NaN,0.000000,0.000000,NaN,1.000000,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,2017-09-10 20:15:46,2017-09-11 15:50:48.500000,2017-09-14 19:52:12,2017-09-22 21:54:31.250000,2017-10-02 00:00:00,NaN,11250.000000,...,1.000000,NaN,1.000000,60.850000,NaN,4.000000,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,2018-01-17 11:59:12,2018-01-17 16:49:49,2018-01-23 17:03:08,2018-02-01 03:17:55,2018-02-14 00:00:00,NaN,24240.000000,...,1.000000,NaN,2.000000,108.160000,NaN,5.000000,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,2018-05-03 13:18:30,2018-05-03 16:56:53,2018-05-07 14:57:00,2018-05-15 00:08:31.500000,2018-05-25 00:00:00,NaN,58475.000000,...,1.000000,NaN,4.000000,189.240000,NaN,5.000000,NaN,NaN,NaN,NaN
max,NaN,NaN,NaN,2018-10-17 17:30:18,2018-09-03 17:40:06,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-11-12 00:00:00,NaN,99990.000000,...,29.000000,NaN,24.000000,13664.080000,NaN,5.000000,NaN,NaN,NaN,NaN


# Step 6: Professional Data Cleaning

The merged dataset is now evaluated and cleaned to ensure it is suitable for downstream analysis.

The cleaning process includes:

- Checking duplicate records
- Handling missing values appropriately
- Verifying data types
- Removing impossible values
- Preparing a clean analytical dataset

Rather than removing all missing values, each column is treated according to its business meaning.

In [14]:
master_df.duplicated().sum()

np.int64(0)

In [15]:
master_df = master_df.drop_duplicates()

print(master_df.shape)

(119143, 36)


In [16]:
missing_summary = pd.DataFrame({
    "Missing Values": master_df.isnull().sum(),
    "Missing %": (
        master_df.isnull().mean() * 100
    ).round(2)
})

missing_summary = (
    missing_summary
    .sort_values("Missing %", ascending=False)
)

missing_summary

,Missing Values,Missing %
review_comment_title,105154,88.26
review_comment_message,68898,57.83
order_delivered_customer_date,3421,2.87
product_category_name,2542,2.13
product_name_lenght,2542,2.13
product_photos_qty,2542,2.13
product_description_lenght,2542,2.13
order_delivered_carrier_date,2086,1.75
review_creation_date,997,0.84
review_score,997,0.84


### Delivery Columns

Missing delivery dates are expected for orders that were cancelled or never shipped.

These values are retained because they represent valid business events rather than data quality issues.

In [17]:
master_df["review_comment_title"] = (
    master_df["review_comment_title"]
    .fillna("No Title")
)

master_df["review_comment_message"] = (
    master_df["review_comment_message"]
    .fillna("No Review")
)

In [18]:
master_df["product_category_name"] = (
    master_df["product_category_name"]
    .fillna("Unknown")
)

In [19]:
numeric_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for col in numeric_cols:
    master_df[col] = master_df[col].fillna(
        master_df[col].median()
    )

In [20]:
master_df.isnull().sum().sort_values(ascending=False)

order_delivered_customer_date    3421
order_delivered_carrier_date     2086
review_answer_timestamp           997
review_creation_date              997
review_score                      997
review_id                         997
shipping_limit_date               833
freight_value                     833
price                             833
order_item_id                     833
product_id                        833
seller_id                         833
order_approved_at                 177
payment_sequential                  3
payment_type                        3
payment_installments                3
payment_value                       3
product_length_cm                   0
review_comment_title                0
review_comment_message              0
product_width_cm                    0
product_height_cm                   0
order_id                            0
product_weight_g                    0
product_photos_qty                  0
product_description_lenght          0
product_name

## Step 7: Feature Engineering

### Feature 1: Delivery Days

Delivery Days measures the number of days between when a customer placed an order and when it was delivered. This feature is useful for evaluating logistics performance and customer experience.

In [21]:
master_df["delivery_days"] = (
    master_df["order_delivered_customer_date"]
    - master_df["order_purchase_timestamp"]
).dt.days

In [22]:
master_df["delivery_days"].describe()

count    115722.000000
mean         12.022589
std           9.454922
min           0.000000
25%           6.000000
50%          10.000000
75%          15.000000
max         209.000000
Name: delivery_days, dtype: float64

In [23]:
master_df["delivery_days"].value_counts().head(10)

delivery_days
7.0     9152
6.0     8149
8.0     8128
9.0     7228
5.0     7013
10.0    6854
11.0    6223
4.0     5825
12.0    5609
13.0    5257
Name: count, dtype: int64

## Feature 2: Delivery Delay

Delivery Delay measures whether an order arrived earlier or later than the estimated delivery date.

Positive values indicate late deliveries.

Negative values indicate early deliveries.

In [24]:
master_df["delivery_delay"] = (
    master_df["order_delivered_customer_date"]
    - master_df["order_estimated_delivery_date"]
).dt.days

In [25]:
master_df["delivery_delay"].describe()

count    115722.000000
mean        -12.048392
std          10.163801
min        -147.000000
25%         -17.000000
50%         -13.000000
75%          -7.000000
max         188.000000
Name: delivery_delay, dtype: float64

In [26]:
master_df["delivery_delay"].value_counts().head(10)

delivery_delay
-14.0    8602
-13.0    7113
-15.0    6414
-7.0     5779
-8.0     5730
-10.0    5564
-9.0     5554
-11.0    5522
-12.0    5458
-16.0    4717
Name: count, dtype: int64

## Feature 3: Total Order Value

Total Order Value represents the total monetary value of each order, including the product price and freight charges.

This feature will later support revenue analysis, customer segmentation, pricing strategy, and machine learning models.

In [28]:
master_df["total_order_value"] = (
    master_df["price"] +
    master_df["freight_value"]
)

In [29]:
master_df["total_order_value"].describe()

count    118310.000000
mean        140.678990
std         191.239906
min           6.080000
25%          55.240000
50%          91.990000
75%         157.615000
max        6929.310000
Name: total_order_value, dtype: float64

In [30]:
master_df["total_order_value"].value_counts().head(10)

total_order_value
77.57     450
73.34     248
35.00     221
67.50     168
56.78     142
107.78    142
116.94    138
45.00     132
34.00     130
105.28    129
Name: count, dtype: int64

## Feature 4: Shipping Ratio

Shipping Ratio measures freight cost as a proportion of product price.

Higher values indicate that shipping contributes a larger share of the item's total cost.

This feature helps identify products or categories with high logistics costs.

In [33]:
master_df["shipping_ratio"] = (
    master_df["freight_value"] /
    master_df["price"]
)

In [34]:
master_df["shipping_ratio"].describe()

count    118310.000000
mean          0.322203
std           0.351861
min           0.000000
25%           0.134787
50%           0.232171
75%           0.394058
max          26.235294
Name: shipping_ratio, dtype: float64

In [35]:
master_df["shipping_ratio"].value_counts().head(10)

shipping_ratio
0.294992    447
0.000000    390
0.224374    248
0.758794    215
0.352705    166
0.198888    139
0.171079    128
0.505017    127
0.302605    122
0.503501    121
Name: count, dtype: int64

# Feature 5: Order Month

Order Month identifies the month in which each order was placed by extracting the month name from the purchase timestamp.

This feature enables time-based analysis and helps uncover seasonal patterns in customer purchasing behavior. Businesses can use it to identify peak sales periods, evaluate monthly revenue trends, compare operational performance across different months, and support inventory and marketing decisions.

Examples of business questions this feature can answer include:

- Which month generated the highest revenue?
- During which months were the most orders placed?
- Do delivery times vary across different months?
- Which months experience higher shipping costs?
- Are customer review scores affected by seasonality?

In [36]:
master_df["order_month"] = (
    master_df["order_purchase_timestamp"]
    .dt.month_name()
)

In [37]:
master_df["order_month"].value_counts()

order_month
August       12802
May          12743
July         12325
March        11858
June         11256
April        11155
February     10180
January       9690
November      9191
December      6646
October       6088
September     5209
Name: count, dtype: int64

## Feature 6: Order Year

Order Year extracts the calendar year from the order purchase timestamp.

Although the Olist dataset covers a limited time period, this feature is useful for longitudinal analysis and is considered a standard practice in time series and business analytics.

Examples of business questions this feature can answer include:

- How does revenue change year over year?
- Has delivery performance improved over time?
- Do customer purchasing patterns vary by year?

In [38]:
master_df["order_year"] = (
    master_df["order_purchase_timestamp"]
    .dt.year
)

In [39]:
master_df["order_year"].value_counts().sort_index()

order_year
2016      409
2017    54549
2018    64185
Name: count, dtype: int64

## Feature 7: Purchase Quarter

Purchase Quarter identifies the financial quarter in which an order was placed.

Quarterly analysis helps businesses evaluate seasonal trends, monitor financial performance, and support strategic planning.

Examples of business questions this feature can answer include:

- Which quarter generated the highest revenue?
- Do delivery times vary across quarters?
- Which quarter experiences the highest customer demand?

In [40]:
master_df["order_quarter"] = (
    "Q" +
    master_df["order_purchase_timestamp"]
    .dt.quarter
    .astype(str)
)

In [41]:
master_df["order_quarter"].value_counts().sort_index()

order_quarter
Q1    31728
Q2    35154
Q3    30336
Q4    21925
Name: count, dtype: int64

## Feature 8: Purchase Weekday

Purchase Weekday identifies the day of the week on which an order was placed.

This feature enables businesses to analyze customer purchasing patterns throughout the week and optimize marketing campaigns, staffing, and operational planning.

Examples of business questions include:

- Which weekday has the highest order volume?
- Are weekend purchases larger than weekday purchases?
- Does delivery performance vary by purchase day?

In [42]:
master_df["purchase_weekday"] = (
    master_df["order_purchase_timestamp"]
    .dt.day_name()
)

In [43]:
master_df["purchase_weekday"].value_counts()

purchase_weekday
Monday       19366
Tuesday      19315
Wednesday    18640
Thursday     17826
Friday       17006
Sunday       14096
Saturday     12894
Name: count, dtype: int64

In [44]:
master_df.to_csv(
    "../data/cleaned/olist_master_cleaned.csv",
    index=False
)